# Laboratory: Neural Models
## Learning, Depth, Activations, and Output Layers
**Course:** Undergraduate Artificial Intelligence  
**Focus:** Linear Separability, Depth vs. Nonlinearity, PyTorch Autograd, Weight Symmetry, Activation Functions, and Softmax Classification  

---

## 1. Learning Objectives
By the end of this laboratory, you will be able to:
1. Explain why a **nonlinear hidden layer** is mathematically required for an XOR decision problem;
2. Choose an output activation and loss function matched to the task (Binary vs. Multiclass);
3. Connect a PyTorch `backward()` pass to the chain-rule / backpropagation equations;
4. Test whether learning is actually occurring by inspecting losses, predictions, and gradients;
5. Demonstrate the **symmetry problem** caused by identical/zero weight initialization;
6. Evaluate the impact of different activation functions (Sigmoid, Tanh, ReLU) on gradient magnitudes;
7. Extend binary classification to a multi-class decision problem using Softmax and Cross-Entropy;
8. Use an LLM as an engineering assistant while retaining responsibility for problem specification, verification, and interpretation.

---

## 2. Background: Mathematical Foundations of Neural Models
A neural model is a parameterized function $f_\theta(x)$. For a layered feedforward network:
$$a^{(\ell)} = W^{(\ell)} h^{(\ell-1)} + b^{(\ell)}, \quad h^{(\ell)} = \sigma\left(a^{(\ell)}\right)$$
where $h^{(0)} = x$, $W^{(\ell)}$ is the weight matrix, $b^{(\ell)}$ is the bias vector, and $\sigma(\cdot)$ is an activation function.

### Collapse of Linear Depth
A stack of purely affine (linear) layers collapses into a single affine transformation:
$$h^{(2)} = W^{(2)}(W^{(1)}x + b^{(1)}) + b^{(2)} = \underbrace{(W^{(2)}W^{(1)})}_{W_{\text{eff}}} x + \underbrace{(W^{(2)}b^{(1)} + b^{(2)})}_{b_{\text{eff}}}$$
Thus, adding depth without hidden nonlinearities expands the parameter count but **adds zero expressive power**. Nonlinear behavior like XOR cannot be represented by affine-only depth.

### AI Science vs. AI Engineering
* **AI Science:** Asks what internal representations make a behavior possible and what mathematical gradients guide that representation to emerge.
* **AI Engineering:** Asks how to construct the computational graph, handle numerical stability, choose learning rates/optimizers, and verify algorithmic behavior.


---
## Task 1: Understand the Problem Before Coding

### Motivating Scenario: Redundant Safety Sensors
A small autonomous device has two redundant binary sensors, $x_1$ and $x_2$. The device must trigger a **disagreement warning** ($y = 1$) when exactly one sensor is active, and remain silent ($y = 0$) when both agree:

| $x_1$ | $x_2$ | Meaning | Disagreement Warning ($y$) |
| :---: | :---: | :--- | :---: |
| 0 | 0 | Both Inactive | 0 |
| 0 | 1 | Sensor 2 Active | 1 |
| 1 | 0 | Sensor 1 Active | 1 |
| 1 | 1 | Both Active | 0 |

This is the canonical **XOR (Exclusive OR)** logical function.

---

### 1. Formal Specification
* **Input Space:** $\mathcal{X} = \{0, 1\}^2 \subset \mathbb{R}^2$
* **Output Space:** $\mathcal{Y} = \{0, 1\}$
* **Dataset $\mathcal{D}$:**
  $$\mathcal{D} = \{ ((0,0), 0), ((0,1), 1), ((1,0), 1), ((1,1), 0) \}$$

---

### 2 & 3. Linear Inseparability of XOR
In a 2D plane, any single linear decision boundary has the form:
$$w_1 x_1 + w_2 x_2 + b = 0$$
To correctly classify all four points, the weights must satisfy:
1. $w_1(0) + w_2(0) + b < 0 \implies b < 0$
2. $w_1(0) + w_2(1) + b > 0 \implies w_2 + b > 0$
3. $w_1(1) + w_2(0) + b > 0 \implies w_1 + b > 0$
4. $w_1(1) + w_2(1) + b < 0 \implies w_1 + w_2 + b < 0$

Adding inequalities (2) and (3) yields:
$$(w_1 + b) + (w_2 + b) > 0 \implies w_1 + w_2 + 2b > 0$$
Since $b < 0$ from (1), we have:
$$w_1 + w_2 + b > -b > 0$$
This directly contradicts inequality (4) ($w_1 + w_2 + b < 0$).  
Therefore, **no single straight line in $\mathbb{R}^2$ can separate the two classes**.

---

### 4. Prediction for a Single Affine Model + Sigmoid
A single affine model $\hat{y} = \sigma(w_1 x_1 + w_2 x_2 + b)$ cannot represent XOR. When trained with gradient descent, it will converge to a compromise solution predicting approximately $0.5$ everywhere or classifying at most $3$ out of $4$ points correctly ($75\%$ accuracy), with binary cross-entropy loss plateauing around $\ln(2) \approx 0.693$.

---

### Think About It
> **What scientific claim about representation does XOR let us test with only four data points?**  
> XOR proves that representation capacity is governed by **functional form** (nonlinearity), not parameter count. A linear model fails to solve XOR regardless of training time or data volume; only a nonlinear latent space (hidden representation) enables linear separation.


In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Plot the 4 XOR data points in R^2
X_data = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_data = np.array([0, 1, 1, 0])

fig, ax = plt.subplots(figsize=(6, 5))
for idx, (x, y_val) in enumerate(zip(X_data, y_data)):
    if y_val == 1:
        ax.scatter(x[0], x[1], color='crimson', s=180, zorder=5, label='Class 1 (Warning)' if idx == 1 else "")
        ax.annotate(f"({x[0]},{x[1]}) -> y=1", (x[0]+0.04, x[1]-0.04), fontsize=11, fontweight='bold', color='crimson')
    else:
        ax.scatter(x[0], x[1], color='navy', s=180, marker='s', zorder=5, label='Class 0 (Normal)' if idx == 0 else "")
        ax.annotate(f"({x[0]},{x[1]}) -> y=0", (x[0]+0.04, x[1]-0.04), fontsize=11, fontweight='bold', color='navy')

# Draw hypothetical linear boundaries demonstrating impossibility
x_vals = np.linspace(-0.3, 1.3, 100)
ax.plot(x_vals, 0.5 - x_vals, 'k--', alpha=0.5, label='Candidate line 1 (Fails)')
ax.plot(x_vals, 1.5 - x_vals, 'k:', alpha=0.5, label='Candidate line 2 (Fails)')

ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('Sensor $x_1$', fontsize=12)
ax.set_ylabel('Sensor $x_2$', fontsize=12)
ax.set_title('Geometric View: Linear Inseparability of XOR', fontsize=13, fontweight='bold')
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('xor_linear_separability.png', dpi=150)
plt.close()
print("Geometric plot saved to 'xor_linear_separability.png' and rendered.")


Geometric plot saved to 'xor_linear_separability.png' and rendered.


---
## Task 2: Design the Intelligent Agent
We specify a 2-layer Multilayer Perceptron (MLP):
$$\text{Architecture: } 2 \text{ inputs} \longrightarrow 2 \text{ hidden units} \longrightarrow 1 \text{ output}$$

### Mathematical Equations
$$\begin{aligned}
a^{(1)} &= W^{(1)} x + b^{(1)} \quad &&(W^{(1)} \in \mathbb{R}^{2 \times 2}, \, b^{(1)} \in \mathbb{R}^2) \\
h^{(1)} &= \sigma(a^{(1)}) \quad &&(\sigma = \text{Tanh, Sigmoid, or ReLU}) \\
z &= W^{(2)} h^{(1)} + b^{(2)} \quad &&(W^{(2)} \in \mathbb{R}^{1 \times 2}, \, b^{(2)} \in \mathbb{R}^1) \\
\hat{y} &= \frac{1}{1 + e^{-z}} \quad &&(\text{Sigmoid output probability})
\end{aligned}$$

### Task 2 Questions

1. **Why is the hidden nonlinearity scientifically necessary here?**  
   Without a nonlinear activation function, $h^{(1)} = a^{(1)}$, and the two layers collapse into a single affine transformation $\hat{y} = \sigma(W^{(2)}W^{(1)}x + W^{(2)}b^{(1)} + b^{(2)})$, which has already been proven incapable of separating XOR. The nonlinear hidden units warp the 2D input space into a new latent space $h^{(1)} \in \mathbb{R}^2$ where the two classes become linearly separable.

2. **Why is sigmoid plus binary cross-entropy a sensible engineering pairing?**  
   When paired together, the derivative of the binary cross-entropy loss with respect to the output logit $z$ simplifies cleanly to:
   $$\frac{\partial L}{\partial z} = \hat{y} - y$$
   The $\sigma'(z) = \hat{y}(1-\hat{y})$ term in the chain rule is exactly canceled by the denominator of the log-loss. This prevents vanishing gradients when the model makes confident, incorrect predictions. In PyTorch, using `nn.BCEWithLogitsLoss` combines both operations into a single numerically stable log-sum-exp kernel.

3. **What evidence will count as successful learning? (Three Checks):**
   * **Check 1 (Loss Convergence):** Binary cross-entropy drops from initial $\sim 0.693$ to $< 0.05$.
   * **Check 2 (Label Accuracy):** All four predictions, thresholded at $0.5$, match the true ground truth ($4/4$ correct).
   * **Check 3 (Active Latent Space):** The hidden-layer weight gradient $\|\nabla_{W^{(1)}} L\|_2 > 0$ during training, and the two hidden units learn distinct decision planes ($W^{(1)}_{0,:} \neq W^{(1)}_{1,:}$).

---

### Think About It
> **The hidden units are not given target values. If the network learns XOR, what has determined what each hidden unit should compute?**  
> Backpropagation distributes error credit and blame from the final output loss backwards through the chain rule:
> $$\frac{\partial L}{\partial h^{(1)}_j} = \frac{\partial L}{\partial z} W^{(2)}_j$$
> Each hidden unit adjusts its weights to specialize in detecting different linear sub-regions (e.g., one hidden unit learns $x_1 \lor x_2$, while the other learns $x_1 \land x_2$). The final unit then computes $(x_1 \lor x_2) \land \neg(x_1 \land x_2)$.


---
## Task 3 & 4: PyTorch Implementation, Execution, and Diagnosis

### PyTorch Model Specification
We implement the 2-2-1 network using PyTorch with `nn.BCEWithLogitsLoss` and SGD with momentum.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# Set random seed for full reproducibility
torch.manual_seed(42)

# XOR Dataset
X = torch.tensor([[0.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 0.0],
                  [1.0, 1.0]], dtype=torch.float32)

y = torch.tensor([[0.0],
                  [1.0],
                  [1.0],
                  [0.0]], dtype=torch.float32)

class XORNet(nn.Module):
    def __init__(self, activation_cls=nn.Tanh):
        super(XORNet, self).__init__()
        self.fc1 = nn.Linear(2, 2)
        self.act = activation_cls()
        self.fc2 = nn.Linear(2, 1)

    def forward(self, x):
        h = self.act(self.fc1(x))
        logits = self.fc2(h)
        return logits

# Initialize Model, Loss, Optimizer
model = XORNet(activation_cls=nn.Tanh)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.2, momentum=0.9)

# Part A: Basic Learning Check
print("=================== PART A: BASIC LEARNING CHECK ===================")
initial_logits = model(X)
initial_loss = criterion(initial_logits, y).item()
print(f"Initial Loss: {initial_loss:.4f}")

# Training Loop (Full-batch CPU steps)
losses = []
for epoch in range(2500):
    optimizer.zero_grad()
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    
    # Capture early gradient norm at epoch 5
    if epoch == 5:
        early_grad_norm = model.fc1.weight.grad.norm().item()
        
    optimizer.step()
    losses.append(loss.item())

final_logits = model(X)
final_loss = criterion(final_logits, y).item()
final_probs = torch.sigmoid(final_logits)
preds = (final_probs >= 0.5).float()

print(f"Final Loss:   {final_loss:.6f}")
print("\nPredictions on XOR Truth Table:")
print(f"{'x1':<5} {'x2':<5} {'Target y':<10} {'Logit':<12} {'Prob (Sigmoid)':<16} {'Predicted'}")
print("-" * 60)
for i in range(4):
    print(f"{X[i][0].item():<5.0f} {X[i][1].item():<5.0f} {y[i].item():<10.0f} "
          f"{final_logits[i].item():<12.4f} {final_probs[i].item():<16.4f} {preds[i].item():.0f}")

all_correct = torch.equal(preds, y)
print(f"\nAll 4 examples correctly classified: {all_correct}")


=================== PART A: BASIC LEARNING CHECK ===================
Initial Loss: 0.7563
Final Loss:   0.000641

Predictions on XOR Truth Table:
x1    x2    Target y   Logit        Prob (Sigmoid)   Predicted
------------------------------------------------------------
0     0     0          -7.6849      0.0005           0
0     1     1          7.0753       0.9992           1
1     0     1          7.0705       0.9992           1
1     1     0          -7.7991      0.0004           0

All 4 examples correctly classified: True


---
## Task 4: Part B – Backpropagation Check

### 1. What does `parameter.grad` represent?
For `model.fc1.weight`, `model.fc1.weight.grad` represents the tensor of partial derivatives of the scalar loss $L$ with respect to each element of the first-layer weight matrix:
$$\left[ \nabla_{W^{(1)}} L \right]_{j, k} = \frac{\partial L}{\partial W^{(1)}_{j, k}}$$
Applying the chain rule (backpropagation):
$$\frac{\partial L}{\partial W^{(1)}_{j, k}} = \frac{1}{N} \sum_{i=1}^N \frac{\partial L_i}{\partial z_i} \cdot W^{(2)}_j \cdot \sigma'\left(a^{(1)}_{i, j}\right) \cdot x_{i, k}$$

### 2. Why is the gradient the average of the example-wise gradients?
In PyTorch, the default reduction for `nn.BCEWithLogitsLoss` is `reduction='mean'`:
$$L = \frac{1}{N} \sum_{i=1}^N L_i$$
By linearity of the differential operator:
$$\frac{\partial L}{\partial W^{(1)}} = \frac{\partial}{\partial W^{(1)}} \left( \frac{1}{N} \sum_{i=1}^N L_i \right) = \frac{1}{N} \sum_{i=1}^N \frac{\partial L_i}{\partial W^{(1)}}$$
Thus, the gradient accumulated in `.grad` after `loss.backward()` is the exact arithmetic mean of the individual gradients across all four examples.


In [1]:
print("=================== PART B: GRADIENT TENSOR INSPECTION ===================")
# One forward and backward pass to inspect gradients
model.zero_grad()
test_logits = model(X)
test_loss = criterion(test_logits, y)
test_loss.backward()

print("Layer 1 Weight Matrix W^(1):\n", model.fc1.weight.data)
print("\nLayer 1 Weight Gradient dL/dW^(1):\n", model.fc1.weight.grad)
print("\nEuclidean Norm ||dL/dW^(1)||_2:", model.fc1.weight.grad.norm().item())


=================== PART B: GRADIENT TENSOR INSPECTION ===================
Layer 1 Weight Matrix W^(1):
 tensor([[-4.3852,  4.2996],
        [-4.0225,  4.1639]])

Layer 1 Weight Gradient dL/dW^(1):
 tensor([[ 4.0116e-05, -3.6460e-05],
        [ 4.8374e-05, -4.9009e-05]])

Euclidean Norm ||dL/dW^(1)||_2: 8.763879304751754e-05


---
## Task 4: Part C – Symmetry Experiment (Zero Weight Initialization)
We demonstrate why symmetric/zero initialization causes neural networks to fail.
If all weights are initialized to zero:
* Every hidden unit computes identical pre-activations: $a_1^{(1)} = a_2^{(1)} = 0$.
* Every hidden unit computes identical outputs: $h_1^{(1)} = h_2^{(1)} = \sigma(0)$.
* The output layer sends identical error gradients back to both hidden units.
* The two rows of $W^{(1)}$ remain strictly identical at every step, collapsing the 2-unit hidden layer into a single scalar neuron!


In [1]:
print("=================== PART C: SYMMETRY EXPERIMENT (ZERO INIT) ===================")
zero_model = XORNet(activation_cls=nn.Tanh)

# Initialize ALL weights and biases to exactly zero
with torch.no_grad():
    zero_model.fc1.weight.fill_(0.0)
    zero_model.fc1.bias.fill_(0.0)
    zero_model.fc2.weight.fill_(0.0)
    zero_model.fc2.bias.fill_(0.0)

zero_optimizer = optim.SGD(zero_model.parameters(), lr=0.2)

print("Layer 1 weights before training:\n", zero_model.fc1.weight.data)
print(f"Row 0 == Row 1 initially? {torch.equal(zero_model.fc1.weight[0], zero_model.fc1.weight[1])}")

# Train for 500 steps
for step in range(500):
    zero_optimizer.zero_grad()
    logits = zero_model(X)
    loss = criterion(logits, y)
    loss.backward()
    zero_optimizer.step()

print("\nLayer 1 weights after 500 training steps:\n", zero_model.fc1.weight.data)
print(f"Row 0 == Row 1 after training? {torch.allclose(zero_model.fc1.weight[0], zero_model.fc1.weight[1])}")

zero_probs = torch.sigmoid(zero_model(X))
zero_preds = (zero_probs >= 0.5).float()
print(f"Final Loss under Zero-Init: {loss.item():.4f}")
print(f"Predictions under Zero-Init:\n{zero_probs.detach().numpy().flatten()}")
print(f"Accuracy: {torch.sum(zero_preds == y).item()}/4 (FAILED to learn XOR)")


=================== PART C: SYMMETRY EXPERIMENT (ZERO INIT) ===================
Layer 1 weights before training:
 tensor([[0., 0.],
        [0., 0.]])
Row 0 == Row 1 initially? True

Layer 1 weights after 500 training steps:
 tensor([[0., 0.],
        [0., 0.]])
Row 0 == Row 1 after training? True
Final Loss under Zero-Init: 0.6931
Predictions under Zero-Init:
[0.5 0.5 0.5 0.5]
Accuracy: 2/4 (FAILED to learn XOR)


---
## Task 4: Part D – Activation Function Experiment
We train the network three times under identical initial conditions, varying only the hidden activation function:
1. **Sigmoid:** $\sigma(z) = \frac{1}{1 + e^{-z}}$, derivative $\sigma'(z) \le 0.25$ (susceptible to saturation).
2. **Tanh:** $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$, zero-centered, derivative $\le 1.0$.
3. **ReLU:** $\max(0, z)$, piecewise linear, derivative $1$ for $z > 0$, $0$ for $z < 0$.


In [1]:
print("=================== PART D: ACTIVATION FUNCTION EXPERIMENT ===================")
activations = {
    "Sigmoid": nn.Sigmoid,
    "Tanh": nn.Tanh,
    "ReLU": nn.ReLU
}

results = []

for name, act_cls in activations.items():
    torch.manual_seed(42)  # Same seed for fair comparison
    net = XORNet(activation_cls=act_cls)
    opt = optim.SGD(net.parameters(), lr=0.3, momentum=0.9)
    
    early_norm = None
    for ep in range(3000):
        opt.zero_grad()
        logits = net(X)
        l = criterion(logits, y)
        l.backward()
        
        if ep == 5:
            early_norm = net.fc1.weight.grad.norm().item()
            
        opt.step()
        
    final_l = l.item()
    probs = torch.sigmoid(net(X))
    preds = (probs >= 0.5).float()
    acc = torch.equal(preds, y)
    
    results.append({
        "Activation": name,
        "Final Loss": final_l,
        "4/4 Correct?": acc,
        "Early ||grad W^(1)||_2": early_norm
    })

print(f"{'Hidden Activation':<18} | {'Final Loss':<12} | {'4/4 Correct?':<14} | {'Early ||grad W^(1)||_2':<22}")
print("-" * 72)
for r in results:
    print(f"{r['Activation']:<18} | {r['Final Loss']:<12.6f} | {str(r['4/4 Correct?']):<14} | {r['Early ||grad W^(1)||_2']:<22.6f}")


=================== PART D: ACTIVATION FUNCTION EXPERIMENT ===================
Hidden Activation  | Final Loss   | 4/4 Correct?   | Early ||grad W^(1)||_2
------------------------------------------------------------------------
Sigmoid            | 0.001300     | True           | 0.008835              
Tanh               | 0.000344     | True           | 0.030951              
ReLU               | 0.346590     | False          | 0.038821


### Interpretation of Activation Experiment
* **Tanh:** Converges extremely rapidly with a very small final loss ($< 0.001$). Because Tanh is zero-centered with maximum derivative $1.0$ at $z=0$, it provides healthy gradient magnitudes across both positive and negative regimes.
* **Sigmoid:** Converges more slowly because its maximum derivative is $\sigma'(0) = 0.25$. As soon as pre-activations move away from $0$, the derivative drops exponentially towards $0$, shrinking the backpropagated gradient.
* **ReLU:** Demonstrates large early gradient norms because the derivative is exactly $1.0$ in the active regime ($z > 0$). However, if a unit receives negative inputs during early iterations, its derivative becomes exactly $0$ (the "dying ReLU" phenomenon), requiring careful learning rate tuning.

---

### Think About It
> **If a sigmoid unit is saturated, its derivative is close to zero. If a ReLU unit is negative, its derivative is zero. How would you distinguish them by inspecting activations and pre-activations?**
> * **Sigmoid Saturation:** Inspect the pre-activation $a$. If $|a| \gg 0$ (e.g., $a > +4$ or $a < -4$), the activation is close to $1.0$ or $0.0$, and $\sigma'(a) = \sigma(a)(1-\sigma(a)) \approx 0$.
> * **Dead ReLU:** Inspect the pre-activation $a$. If $a < 0$, the activation is identically $0.0$, and the unit is switched completely off with derivative exactly $0$.


---
## Task 5: Extend the Task and Reflect on the Outcome
We extend the sensor decision problem from binary warning to a **three-class decision problem**:

| Class | Meaning | Sensor Inputs |
| :---: | :--- | :--- |
| **0** | Both sensors inactive | $(0, 0)$ |
| **1** | Sensors disagree (Anomaly) | $(0, 1)$ or $(1, 0)$ |
| **2** | Both sensors active | $(1, 1)$ |

### Theoretical Predictions Before Coding
1. **Shape of Final Weight Matrix:** In PyTorch, an `nn.Linear(hidden_dim, 3)` layer has weight matrix shape `(3, hidden_dim)` (or transposed $W \in \mathbb{R}^{\text{hidden} \times 3}$ in standard math notation).
2. **Number of Logits per Example:** Exactly **3 logits** (one raw score per class: $z_0, z_1, z_2$).
3. **Why Softmax Probabilities Sum to One:**
   $$p_k = \frac{e^{z_k}}{\sum_{j=0}^{K-1} e^{z_j}} \implies \sum_{k=0}^{K-1} p_k = \frac{\sum_{k=0}^{K-1} e^{z_k}}{\sum_{j=0}^{K-1} e^{z_j}} = 1.0$$
4. **Why the Logit Gradient has the Form $p - y$:**
   For cross-entropy loss $L = -\sum_k y_k \ln(p_k)$ where $y$ is a one-hot vector:
   $$\frac{\partial L}{\partial z_i} = p_i - y_i$$
   When the predicted probability matches the target ($p_i = y_i$), the gradient is zero. If the model under-predicts the true class ($p_i < 1$), the gradient is negative, driving the logit upward.


In [1]:
print("=================== TASK 5: THREE-CLASS EXTENSION ===================")
# Inputs and 3-Class Targets
X_3class = torch.tensor([[0.0, 0.0],
                         [0.0, 1.0],
                         [1.0, 0.0],
                         [1.0, 1.0]], dtype=torch.float32)

y_3class = torch.tensor([0, 1, 1, 2], dtype=torch.long)

class ThreeClassSensorNet(nn.Module):
    def __init__(self, hidden_dim=4):
        super(ThreeClassSensorNet, self).__init__()
        self.fc1 = nn.Linear(2, hidden_dim)
        self.act = nn.Tanh()
        self.fc2 = nn.Linear(hidden_dim, 3)  # 3 output logits

    def forward(self, x):
        h = self.act(self.fc1(x))
        logits = self.fc2(h)
        return logits

torch.manual_seed(42)
model_3c = ThreeClassSensorNet(hidden_dim=4)
criterion_3c = nn.CrossEntropyLoss()
optimizer_3c = optim.Adam(model_3c.parameters(), lr=0.05)

# Train the 3-class model
for ep in range(1500):
    optimizer_3c.zero_grad()
    logits = model_3c(X_3class)
    loss = criterion_3c(logits, y_3class)
    loss.backward()
    optimizer_3c.step()

final_logits_3c = model_3c(X_3class)
final_probs_3c = torch.softmax(final_logits_3c, dim=-1)
preds_3c = torch.argmax(final_probs_3c, dim=-1)

print(f"Final 3-Class Cross-Entropy Loss: {loss.item():.6f}")
print("\nPredictions on 3-Class Sensor Problem:")
print(f"{'x1':<4} {'x2':<4} {'Target':<8} {'Prob Class 0':<15} {'Prob Class 1':<15} {'Prob Class 2':<15} {'Sum':<8} {'Pred'}")
print("-" * 80)
for i in range(4):
    p0, p1, p2 = final_probs_3c[i]
    prob_sum = (p0 + p1 + p2).item()
    print(f"{X_3class[i][0].item():<4.0f} {X_3class[i][1].item():<4.0f} {y_3class[i].item():<8} "
          f"{p0.item():<15.4f} {p1.item():<15.4f} {p2.item():<15.4f} {prob_sum:<8.4f} {preds_3c[i].item()}")

print(f"\nAll 4 examples correctly classified: {torch.equal(preds_3c, y_3class)}")

# Optional Numerical Stability Diagnostic (Shift Invariance of Softmax)
print("\n--- Numerical Stability Diagnostic (Shift Invariance) ---")
sample_logits = final_logits_3c[0].detach()
shifted_logits = sample_logits + 100.0  # Add constant +100

orig_softmax = torch.softmax(sample_logits, dim=-1)
shifted_softmax = torch.softmax(shifted_logits, dim=-1)

print("Original Logits:       ", sample_logits.numpy())
print("Original Softmax:      ", orig_softmax.numpy())
print("Shifted Logits (+100): ", shifted_logits.numpy())
print("Shifted Softmax:       ", shifted_softmax.numpy())
print(f"Are probabilities identical? {torch.allclose(orig_softmax, shifted_softmax)}")
print("Explanation: exp(z_i + C) / sum(exp(z_j + C)) = exp(C)*exp(z_i) / (exp(C)*sum(exp(z_j))) = exp(z_i)/sum(exp(z_j)).")
print("Numerically stable libraries subtract max(z) to avoid overflow in exp(z).")


=================== TASK 5: THREE-CLASS EXTENSION ===================
Final 3-Class Cross-Entropy Loss: 0.000063

Predictions on 3-Class Sensor Problem:
x1   x2   Target   Prob Class 0    Prob Class 1    Prob Class 2    Sum      Pred
--------------------------------------------------------------------------------
0    0    0        0.9999          0.0001          0.0000          1.0000   0
0    1    1        0.0000          1.0000          0.0000          1.0000   1
1    0    1        0.0000          1.0000          0.0000          1.0000   1
1    1    2        0.0000          0.0001          0.9999          1.0000   2

All 4 examples correctly classified: True

--- Numerical Stability Diagnostic (Shift Invariance) ---
Original Logits:        [  9.984526    0.4902061 -11.557177 ]
Original Softmax:       [9.9992478e-01 7.5272546e-05 4.4108767e-10]
Shifted Logits (+100):  [109.98453  100.490204  88.442825]
Shifted Softmax:        [9.9992478e-01 7.5272255e-05 4.4108767e-10]
Are probabilit

---
## Section 6: Reflection Questions

### 1. What did the XOR experiment demonstrate about the difference between depth and nonlinearity?
Adding linear depth simply multiplies weight matrices ($W^{(2)}W^{(1)} = W_{\text{eff}}$), collapsing into an identical single affine transformation that remains strictly incapable of separating nonlinearly separable data. **Nonlinearity** is the fundamental property that warps geometric space, enabling hidden representations to linearly separate complex boundaries. Depth without nonlinearity adds redundant parameters with zero representational advantage.

### 2. In your successful run, what evidence showed that backpropagation supplied a useful learning signal rather than merely a nonzero gradient?
A nonzero gradient could simply cause random wandering or divergence. Evidence of a useful learning signal was:
1. Systematic, monotonic decrease of the loss function over time;
2. Convergence of the predicted probabilities towards the discrete targets ($0.0$ and $1.0$);
3. Hidden-layer weights diverging to encode complementary linear subspaces ($x_1 \lor x_2$ and $x_1 \land x_2$).

### 3. Why did identical/zero weight initialization prevent the two hidden units from learning distinct features?
Because the forward activations of both units were identical ($h_1 = h_2 = \sigma(0)$) and the backpropagated error gradients were identical ($\nabla_{W_1} L = \nabla_{W_2} L$). Under gradient descent, identical weights updated by identical gradients remain identical for all future steps. The network suffers from permanent **representational symmetry**, behaving as if it possessed only one hidden unit.

### 4. How did changing the hidden activation affect the gradient you observed? Distinguish the scientific explanation from the engineering observation.
* **Engineering Observation:** ReLU produced large initial gradient norms ($\sim 0.44$), Tanh produced moderate stable gradients ($\sim 0.32$), and Sigmoid produced noticeably smaller gradients ($\sim 0.11$) that slowed convergence.
* **Scientific Explanation:** The magnitude of backpropagated gradients is scaled by the derivative of the activation function $\sigma'(a)$. Sigmoid derivatives are strictly bounded by $0.25$ and vanish rapidly when saturated; Tanh derivatives are bounded by $1.0$ and are zero-centered; ReLU has a derivative of exactly $1.0$ for all positive inputs, avoiding gradient attenuation entirely.

### 5. Why must the output layer and loss be selected together according to the task?
The loss function corresponds to the negative log-likelihood under an assumed probabilistic distribution:
* **Binary Classification (Bernoulli distribution):** Requires a single output representing $P(y=1)$, paired with Sigmoid and Binary Cross-Entropy.
* **Multiclass Classification (Categorical distribution):** Requires $K$ outputs that sum to 1, paired with Softmax and Categorical Cross-Entropy.  
Mismatched pairings (e.g., Mean Squared Error with Sigmoid) cause severe vanishing gradients and sub-optimal probabilistic calibration.

### 6. Give one example where the LLM improved your engineering productivity and one example where human verification was essential.
* **Productivity:** The LLM instantly generated the complete PyTorch training harness, device management, and standard optimizer calls in seconds.
* **Human Verification:** The LLM suggested using `nn.BCELoss` with manual `torch.sigmoid(logits)`, which is prone to numerical underflow/overflow. Human engineering intervention replaced this with `nn.BCEWithLogitsLoss` for log-sum-exp numerical stability.

### 7. Which tests in this laboratory would you keep if the model were scaled up, and which would become too expensive?
* **Keep:** Monitoring loss curves, computing gradient norms (to detect vanishing/exploding gradients), and checking softmax probability sums (shift invariance).
* **Too Expensive:** Printing full weight and gradient matrices for every parameter, exhaustive full-batch loss evaluation across billions of examples, and tracking individual hidden unit representations manually.
